# U13-2 — `bar_off_floor` 검토시간 비교

U13-1 자동 제안이 완전 수동 검토보다 실제 검토시간을 줄이는지 확인합니다. 새 모델을 학습하거나 추론하지 않고 저장된 원판 track과 고정 threshold `0.01`을 재사용합니다.

이 실험은 서로 다른 6영상을 사용하는 소규모 탐색 실험입니다. 같은 영상을 두 번 보면 정답 프레임을 기억하는 문제가 있어 교차 실험은 하지 않습니다. 따라서 결과는 인과관계나 일반화의 증명이 아니라, 자동 제안을 **선택적 수동 검토 보조 기능으로 채택할 근거**입니다.

In [ ]:
from math import ceil
from pathlib import Path
from time import perf_counter
import os

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from snatch_analysis.phases import (
    propose_bar_off_floor_event,
    summarize_bar_off_floor_timing_experiment,
)
from snatch_analysis.video import iter_video_frames

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요.')
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

## 1. 고정 실험 계약

- 완전 수동: A08, B02, C03
- 자동 제안 보조: A02, B04, C04
- 성공 조건: 6/6 완료, 보조 방식 중앙값이 수동보다 25% 이상 짧음, A/B/C 중 2집단 이상에서 보조 방식이 더 빠름
- 시간 범위: 준비 셀의 track·프레임 사전 로딩은 제외하고, 검토 화면 표시부터 확정까지의 사용자 상호작용을 측정
- 자동 제안은 `phase_events.csv`를 수정하지 않으며, 확정 프레임은 이 실험의 ignored CSV에만 기록

In [ ]:
U13_TIMING_TRIALS = pd.DataFrame(
    [
        ('a08_sn_f_01', 'A', 'manual'),
        ('a02_sn_f_01', 'A', 'assisted'),
        ('b02_sn_f_01', 'B', 'manual'),
        ('b04_sn_f_01', 'B', 'assisted'),
        ('c03_sn_f_01', 'C', 'manual'),
        ('c04_sn_f_01', 'C', 'assisted'),
    ],
    columns=['video_id', 'group', 'method'],
)

# 한 번에 한 영상만 입력합니다. 권장 순서는 위 표의 위에서 아래입니다.
U13_TIMING_VIDEO_ID = None
U13_TIMING_CONFIRMED_FRAME = None
U13_TIMING_FAILURE_REASON = ''
OVERWRITE_U13_TIMING_TRIAL = False

U13_UPWARD_SPEED_THRESHOLD = 0.01
U13_PERSISTENCE_FRAMES = 2
U13_MAX_MISSING_GAP_FRAMES = 2
U13_MANUAL_REVIEW_SECONDS = 1.5
U13_ASSISTED_REVIEW_RADIUS = 4

U13_MANIFEST_PATH = PROJECT_ROOT / 'data/metadata/videos.csv'
U13_TRACK_ROOT = PROJECT_ROOT / 'artifacts/u11_primary_plate_selection/task2/inputs'
U13_TIMING_RESULT_PATH = PROJECT_ROOT / 'artifacts/u13_bar_off_floor/review_timing.csv'
display(U13_TIMING_TRIALS)

## 2. 한 실험 준비

`U13_TIMING_VIDEO_ID`를 하나 선택한 뒤 설정 셀부터 이 셀까지 실행합니다. 준비 셀은 비교 시간에 포함하지 않기 위해 필요한 track과 축소 프레임을 미리 읽지만, 검토 화면이나 자동 제안 프레임은 아직 보여주지 않습니다.

In [ ]:
_u13_timing_ready = False
if U13_TIMING_VIDEO_ID is None:
    print('실험 준비 대기: U13_TIMING_VIDEO_ID를 입력하세요.')
else:
    trial_ids = set(U13_TIMING_TRIALS['video_id'])
    if U13_TIMING_VIDEO_ID not in trial_ids:
        raise ValueError('고정된 U13_TIMING_TRIALS 영상 중 하나를 입력하세요.')

    manifest = pd.read_csv(U13_MANIFEST_PATH)
    if manifest['video_id'].duplicated().any():
        raise ValueError('Manifest video_id가 중복되었습니다.')
    metadata_by_video = manifest.set_index('video_id', drop=False)
    if U13_TIMING_VIDEO_ID not in metadata_by_video.index:
        raise ValueError(f'Manifest에 없는 영상: {U13_TIMING_VIDEO_ID}')

    _u13_trial = U13_TIMING_TRIALS.set_index('video_id').loc[U13_TIMING_VIDEO_ID]
    _u13_metadata = metadata_by_video.loc[U13_TIMING_VIDEO_ID]
    if str(_u13_metadata['read_status']) != 'ok':
        raise ValueError(f'읽을 수 없는 영상: {U13_TIMING_VIDEO_ID}')
    _u13_video_path = PROJECT_ROOT / str(_u13_metadata['path'])
    _u13_frame_count = int(_u13_metadata['frame_count'])
    _u13_proposal = None

    if _u13_trial['method'] == 'assisted':
        _u13_track_path = (
            U13_TRACK_ROOT / U13_TIMING_VIDEO_ID / 'plate_track.csv'
        )
        if not _u13_track_path.exists():
            raise FileNotFoundError(f'저장된 plate track 없음: {_u13_track_path}')
        _u13_proposal = propose_bar_off_floor_event(
            pd.read_csv(_u13_track_path),
            upward_speed_threshold=U13_UPWARD_SPEED_THRESHOLD,
            persistence_frames=U13_PERSISTENCE_FRAMES,
            max_missing_gap_frames=U13_MAX_MISSING_GAP_FRAMES,
        )
        if _u13_proposal['status'] != 'ok':
            raise RuntimeError(
                '자동 제안 실패: ' + str(_u13_proposal['reason'])
            )
        _u13_center = int(_u13_proposal['frame_index'])
        _u13_review_indices = range(
            max(0, _u13_center - U13_ASSISTED_REVIEW_RADIUS),
            min(_u13_frame_count, _u13_center + U13_ASSISTED_REVIEW_RADIUS + 1),
        )
    else:
        _u13_review_indices = range(
            min(
                _u13_frame_count,
                ceil(float(_u13_metadata['fps']) * U13_MANUAL_REVIEW_SECONDS) + 1,
            )
        )

    _u13_wanted = set(_u13_review_indices)
    _u13_review_frames = {}
    for frame_index, _, frame in iter_video_frames(_u13_video_path):
        if frame_index in _u13_wanted:
            scale = min(1.0, 240 / frame.shape[1])
            size = (int(frame.shape[1] * scale), int(frame.shape[0] * scale))
            _u13_review_frames[frame_index] = cv2.cvtColor(
                cv2.resize(frame, size), cv2.COLOR_BGR2RGB
            )
        if frame_index >= max(_u13_wanted):
            break
    missing_frames = _u13_wanted - set(_u13_review_frames)
    if missing_frames:
        raise RuntimeError(f'검토 프레임 읽기 실패: {sorted(missing_frames)}')

    if U13_TIMING_RESULT_PATH.exists():
        existing_results = pd.read_csv(U13_TIMING_RESULT_PATH)
        already_saved = existing_results['video_id'].astype(str).eq(
            U13_TIMING_VIDEO_ID
        ).any()
        if already_saved and not OVERWRITE_U13_TIMING_TRIAL:
            raise ValueError(
                '이미 저장된 영상입니다. 다시 측정할 때만 '
                'OVERWRITE_U13_TIMING_TRIAL=True로 바꾸세요.'
            )

    _u13_timing_ready = True
    print(
        f'준비 완료: {U13_TIMING_VIDEO_ID} | method={_u13_trial["method"]} | '
        f'검토 프레임 {len(_u13_review_frames)}개 사전 로딩'
    )

## 3. 타이머 시작과 육안 검토

`RUN_U13_TIMING_REVIEW=True`로 바꾸고 아래 셀을 **한 번만** 실행합니다. 셀이 화면을 표시하는 순간부터 타이머가 시작됩니다.

- `manual`: 첫 1.5초의 모든 프레임에서 원판이 처음 상승한 프레임을 찾습니다.
- `assisted`: 자동 제안 ±4프레임만 보고 제안을 그대로 확정하거나 수정합니다.

확인 즉시 설정 셀의 `U13_TIMING_CONFIRMED_FRAME`에 첫 상승 프레임을 적으세요. 판단할 수 없으면 프레임은 `None`으로 두고 `U13_TIMING_FAILURE_REASON`에 사유를 적습니다.

In [ ]:
RUN_U13_TIMING_REVIEW = False

if not RUN_U13_TIMING_REVIEW:
    print('검토 대기: 준비 완료 후 RUN_U13_TIMING_REVIEW=True로 바꾸세요.')
elif not _u13_timing_ready:
    raise RuntimeError('먼저 실험 준비 셀을 실행하세요.')
else:
    _u13_review_started_at = perf_counter()
    columns = 5 if _u13_trial['method'] == 'assisted' else 6
    rows = ceil(len(_u13_review_frames) / columns)
    figure, axes = plt.subplots(rows, columns, figsize=(columns * 3, rows * 2.5))
    axes = [axes] if rows * columns == 1 else list(axes.flat)
    for axis, (frame_index, frame) in zip(axes, _u13_review_frames.items()):
        axis.imshow(frame)
        axis.set_title(f'frame {frame_index}')
        axis.axis('off')
    for axis in axes[len(_u13_review_frames):]:
        axis.axis('off')
    figure.suptitle(
        f'{U13_TIMING_VIDEO_ID} | method={_u13_trial["method"]}', fontsize=14
    )
    plt.tight_layout()
    plt.show()
    if _u13_proposal is None:
        print('완전 수동 검토: 자동 제안을 사용하지 않습니다.')
    else:
        print(f'자동 제안 frame: {_u13_proposal["frame_index"]}')

## 4. 확정과 저장

설정 셀에서 확정 프레임 또는 실패 사유를 입력한 뒤 `SAVE_U13_TIMING_RESULT=True`로 바꾸고 실행합니다. 동일 영상을 덮어쓸 때만 `OVERWRITE_U13_TIMING_TRIAL=True`를 사용합니다. 저장 후 다음 영상으로 넘어갈 때는 설정 셀부터 반복합니다.

In [ ]:
SAVE_U13_TIMING_RESULT = False

if not SAVE_U13_TIMING_RESULT:
    print('저장 대기: 검토 직후 SAVE_U13_TIMING_RESULT=True로 바꾸세요.')
elif not _u13_timing_ready or globals().get('_u13_review_started_at') is None:
    raise RuntimeError('준비 셀과 타이머 시작 셀을 먼저 실행하세요.')
else:
    review_seconds = perf_counter() - _u13_review_started_at
    failure_reason = U13_TIMING_FAILURE_REASON.strip()
    if failure_reason:
        status = 'failed'
        confirmed_frame = pd.NA
    else:
        if isinstance(U13_TIMING_CONFIRMED_FRAME, bool) or not isinstance(
            U13_TIMING_CONFIRMED_FRAME, int
        ):
            raise ValueError('확정한 첫 상승 frame index를 정수로 입력하세요.')
        if not 0 <= U13_TIMING_CONFIRMED_FRAME < _u13_frame_count:
            raise ValueError(f'frame 범위는 0..{_u13_frame_count - 1}입니다.')
        status = 'complete'
        confirmed_frame = U13_TIMING_CONFIRMED_FRAME

    proposal_frame = (
        pd.NA if _u13_proposal is None else int(_u13_proposal['frame_index'])
    )
    correction_frames = (
        pd.NA
        if pd.isna(proposal_frame) or pd.isna(confirmed_frame)
        else abs(int(confirmed_frame) - int(proposal_frame))
    )
    result_row = pd.DataFrame(
        [{
            'video_id': U13_TIMING_VIDEO_ID,
            'group': str(_u13_trial['group']),
            'method': str(_u13_trial['method']),
            'proposal_frame': proposal_frame,
            'confirmed_frame': confirmed_frame,
            'correction_frames': correction_frames,
            'review_seconds': round(review_seconds, 3),
            'status': status,
            'failure_reason': failure_reason,
        }]
    )

    if U13_TIMING_RESULT_PATH.exists():
        timing_results = pd.read_csv(U13_TIMING_RESULT_PATH)
        timing_results = timing_results.loc[
            ~timing_results['video_id'].astype(str).eq(U13_TIMING_VIDEO_ID)
        ]
        timing_results = pd.concat([timing_results, result_row], ignore_index=True)
    else:
        timing_results = result_row
    timing_results = timing_results.sort_values(['group', 'method', 'video_id'])

    U13_TIMING_RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = U13_TIMING_RESULT_PATH.with_suffix('.tmp')
    try:
        timing_results.to_csv(temporary_path, index=False)
        temporary_path.replace(U13_TIMING_RESULT_PATH)
    finally:
        temporary_path.unlink(missing_ok=True)
    _u13_review_started_at = None
    display(result_row)
    print(f'저장 완료: {U13_TIMING_RESULT_PATH}')

## 5. 진행도와 최종 판정

저장된 CSV만 읽으므로 모델이나 영상 처리를 다시 실행하지 않습니다. 6개가 완료되기 전에는 `incomplete`가 정상입니다. 모든 수치를 확인한 뒤에만 자동 제안 보조 기능 채택 여부를 해석합니다.

In [ ]:
if not U13_TIMING_RESULT_PATH.exists():
    print('아직 저장된 시간 측정 결과가 없습니다.')
else:
    timing_results = pd.read_csv(U13_TIMING_RESULT_PATH)
    display(timing_results)
    timing_summary = summarize_bar_off_floor_timing_experiment(timing_results)
    display(pd.Series(timing_summary, name='U13-2 timing summary').to_frame())

    complete_results = timing_results.loc[
        timing_results['status'].astype(str).eq('complete')
    ]
    if not complete_results.empty:
        group_comparison = complete_results.pivot(
            index='group', columns='method', values='review_seconds'
        )
        display(group_comparison)

    if timing_summary['status'] == 'incomplete':
        print(
            f'측정 진행 중: {timing_summary["complete_count"]}/'
            f'{timing_summary["expected_count"]}'
        )
    elif timing_summary['time_gate_pass']:
        print('시간 gate PASS: 자동 제안을 선택적 수동 검토 보조로 채택할 근거가 생겼습니다.')
    else:
        print('시간 gate HOLD: 자동 제안은 정확도 실험 결과만 유지하고 UX 채택은 보류합니다.')

## 해석 한계

- 6영상·1명의 검토자 결과이므로 일반화나 인과관계를 주장하지 않습니다.
- 수동과 보조 방식에 서로 다른 영상을 배정했으므로 영상 난이도 차이가 남아 있습니다.
- 사전 계산된 detector·tracker·자동 제안 시간은 제외했고, 검토 화면 표시와 사람의 확인·입력 시간만 측정합니다.
- 수치 gate가 통과해도 자동 정답 저장에는 사용하지 않습니다. 사람이 확정한 프레임만 별도 실험 결과로 남깁니다.